# 插件机制

学习目标：为文本处理程序发现并加载插件，使用入口点连接分发包与宿主，并区分元数据、加载和调用阶段的错误。

前置知识：模块与包、函数对象、Callable 类型标注、异常链、文件路径、with、子进程、TOML、wheel 构建与 pytest 断言。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

构建与测试使用 build、setuptools、wheel 和 pytest，版本见 [requirements.txt](requirements.txt)。

实验只构建本地小包，不下载依赖或发布。构建使用临时源码副本，安装目标和运行目录也在自动清理的临时目录中；插件只在子进程导入。

配套脚本：位于 [scripts/33-plugin-system/](scripts/33-plugin-system/)。

（1）[pyproject.toml](scripts/33-plugin-system/pyproject.toml)：构建配置及四条插件入口记录。

（2）[src/study\_formatters/](scripts/33-plugin-system/src/study_formatters/)：正常文本插件、不可调用对象、受控调用错误和故意缺失内部依赖的反例。

（3）[plugin\_host.py](scripts/33-plugin-system/plugin_host.py)：发现、元数据校验、加载及调用边界。

（4）[test\_plugin\_host.py](scripts/33-plugin-system/test_plugin_host.py)：宿主行为测试，部分入口记录为明确构造的模拟数据。

## 1 通过模块名称动态导入

插件让宿主通过约定接入额外功能。动态导入是其中的加载手段：importlib.import\_module 接收模块名称字符串，返回指定模块；使用相对名称时还要提供作为解析起点的 package 参数。

本章先用已有的 math 模块观察这种调用，再把同样的导入机制用于文本插件。动态导入沿用 Python 的导入规则，并不建立独立的执行环境。

In [1]:
import importlib
from pathlib import Path

chapter_dir = Path("scripts/33-plugin-system").resolve()
module_name = "math"
module = importlib.import_module(module_name)
print(module.__name__, module.sqrt(81))  # math 9.0。
assert module.sqrt(81) == 9.0
# 返回的是模块对象，可以继续读取属性或调用函数。

math 9.0


## 2 模块缓存与运行中新增文件

导入首先查询 sys.modules。已有模块通常直接复用；修改源码文件后再次调用 import\_module，不会自动执行新文件内容。

importlib.invalidate\_caches 通知查找器清理其缓存，使运行中新建或安装的模块可以被发现。它不会清除 sys.modules，也不负责更新已有函数引用。reload 会重新执行模块代码，但已有外部引用未必同步，因此本章用新进程开始每次插件实验。

下面故意让临时模块在顶层打印一句话，用来观察导入执行；正式插件应避免在顶层启动业务任务。子进程使用当前解释器，环境映射来自副本；-B 和 PYTHONDONTWRITEBYTECODE 禁止写字节码缓存。

In [2]:
import os
import subprocess
import sys
import tempfile

child_env = os.environ.copy()
child_env.update({
    "PYTHONPATH": "",
    "PYTHONDONTWRITEBYTECODE": "1",
    "PYTHONUTF8": "1",
    "PYTHONIOENCODING": "utf-8",
    "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1",
})
cache_source = r"""
import importlib
from pathlib import Path

path = Path("temporary_plugin.py")
path.write_text('print("模块顶层运行")\nvalue = 1\n', encoding="utf-8")
first = importlib.import_module("temporary_plugin")
path.write_text("value = 200\n", encoding="utf-8")
importlib.invalidate_caches()
second = importlib.import_module("temporary_plugin")
assert first is second and second.value == 1
print(first is second, second.value)
Path("added_plugin.py").write_text("value = 3\n", encoding="utf-8")
importlib.invalidate_caches()
print(importlib.import_module("added_plugin").value)
"""
with tempfile.TemporaryDirectory(dir=chapter_dir) as directory:
    result = subprocess.run(
        [sys.executable, "-B", "-c", cache_source],
        cwd=directory, env=child_env, capture_output=True,
        encoding="utf-8", check=True, timeout=30,
    )
    print(result.stdout, end="")
    assert result.stdout == "模块顶层运行\nTrue 1\n3\n"
assert not Path(directory).exists()
# 顶层输出只出现一次；失效查找缓存后，旧模块仍保留 value = 1。

模块顶层运行
True 1
3


## 3 发现方式与入口点的职责

发现只产生候选记录，加载才解析并导入对象，调用才把业务输入交给对象。这三个阶段分开后，宿主可以先检查名称冲突，再决定加载哪一个插件。

| 方式 | 中文名称／含义 | 发现约定 |
| --- | --- | --- |
| naming convention | 命名约定 | 按共同前缀筛选模块名称 |
| namespace package | 命名空间包 | 多个分发包向同一包命名空间提供子模块 |
| entry point | 入口点 | 分发元数据登记接口组、插件名和对象引用 |

pkgutil.iter\_modules 可以列出指定路径中的模块名称；省略路径时会查看 sys.path。本例只列出配套 src，不导入其中的包，也不扫描共享环境。命名空间方案可把包的 \_\_path\_\_ 传给它，列出该命名空间中的直接子模块；这里重点实践元数据方案。

In [3]:
import pkgutil

candidates = [
    item.name
    for item in pkgutil.iter_modules([str(chapter_dir / "src")])
    if item.name.startswith("study_")
]
print(candidates)  # ['study_formatters']，这里只发现名称。
assert candidates == ["study_formatters"]
assert "study_formatters" not in sys.modules

['study_formatters']


## 4 在分发包中登记入口点

一个分发包可以登记多个入口点；分发包名称、导入包名称和插件名称各自承担不同职责。本例分发名为 study-plugin-demo，导入包为 study\_formatters。

| 属性 | 中文名称／含义 | 本例 |
| --- | --- | --- |
| group | 接口组，由宿主约定 | study\_plugin\_demo.formatters.v1 |
| name | 组内插件名称 | upper |
| value | 对象引用 | study\_formatters.good:format\_text |

value 中冒号左侧是模块路径，右侧是模块中的属性路径。自定义组通过 project.entry-points 下的表登记，带点的组名需要在 TOML 中加引号。本例采用本地演示组名；正式项目应使用归属于宿主项目的名称前缀，减少冲突。

v1 是本宿主对接口版本的约定，入口点系统不会自动替我们检查兼容性。与 console\_scripts 不同，这个自定义组供宿主查询，不要求安装工具生成命令包装程序。

In [4]:
import tomllib

with (chapter_dir / "pyproject.toml").open("rb") as stream:
    project_config = tomllib.load(stream)
group = "study_plugin_demo.formatters.v1"
entry_mapping = project_config["project"]["entry-points"][group]
for name, reference in sorted(entry_mapping.items()):
    print(name, "->", reference)
assert set(entry_mapping) == {"upper", "broken", "reject", "dependency"}
# upper 正常；broken 不可调用；reject 调用失败；dependency 加载失败。

broken -> study_formatters.bad:not_a_function
dependency -> study_formatters.dependency:format_text
reject -> study_formatters.bad:reject_text
upper -> study_formatters.good:format_text


## 5 约定宿主的文本接口

本例接口接收一个字符串位置参数，返回字符串，允许空字符串。Callable[[str], str] 的第一项列表描述参数类型，第二项描述返回类型；type Formatter 为这个约定命名。

如果接口需要多个方法或具名属性，可以使用 Protocol 描述结构。本章只有一个调用动作，所以使用 Callable。类型标注不会自动执行运行时检查；即使 callable 返回 True，也不能保证参数数量、返回类型或业务行为符合约定。

In [5]:
from collections.abc import Callable

type Formatter = Callable[[str], str]
formatter: Formatter = str.upper
print(formatter("Py插件"))  # PY插件。
assert formatter("") == ""
assert formatter("Py插件") == "PY插件"
print(callable(formatter), callable(42))  # True False。
# str.upper 只是本节的接口示例，尚未查询或加载分发包入口。

PY插件
True False


## 6 在临时副本中构建 wheel

配套 pyproject.toml 声明 Python >=3.12、空运行依赖及固定构建工具。正常插件只调用字符串的 upper；其余入口是故意保留的错误样例，缺失依赖名称不对应需要安装的真实项目。

python -m build --wheel --no-isolation 使用当前环境中的构建后端生成 wheel，不另建环境或安装构建依赖。先复制源码，再在副本中构建，使 build 和 egg-info 等中间目录随 TemporaryDirectory 一起删除。下面只把生成的 wheel 字节保留在内存中。

In [6]:
import shutil

with tempfile.TemporaryDirectory(dir=chapter_dir) as directory:
    build_root = Path(directory)
    build_copy = build_root / "project"
    build_copy.mkdir()
    shutil.copy2(chapter_dir / "pyproject.toml", build_copy)
    shutil.copytree(chapter_dir / "src", build_copy / "src")
    artifacts = build_root / "artifacts"
    result = subprocess.run(
        [sys.executable, "-B", "-m", "build", "--wheel", "--no-isolation",
         "--outdir", str(artifacts), str(build_copy)],
        cwd=build_root, env=child_env, capture_output=True,
        encoding="utf-8", check=True, timeout=120,
    )
    (wheel_path,) = artifacts.glob("*.whl")
    wheel_name = wheel_path.name
    wheel_bytes = wheel_path.read_bytes()
    print(wheel_name)
assert not build_root.exists()
# 应得到 study_plugin_demo 的 0.1.0 wheel；构建副本和产物目录已清理。

study_plugin_demo-0.1.0-py3-none-any.whl


## 7 安装到指定目录并在新进程中运行

pip install --target 指定安装目录；--no-index 禁止查询索引，--no-deps 禁止安装依赖。--target 不会创建新环境，也不会自动修改 Notebook 的模块搜索路径。

下面定义实验函数，每次调用都把同一个 wheel 安装到新临时目录。宿主和测试脚本复制到 outside，子进程从那里启动，PYTHONPATH 只增加安装目标；插件源码 src 不进入搜索路径。

source 表示要在这个子进程中执行的 Python 代码字符串。函数先导入宿主，并把安装位置命名为 target，供后续代码使用。check=True 要求安装和执行均成功；子进程结束后临时目录自动清理。

In [7]:
def run_installed(source: str) -> str:
    """临时安装当前 wheel，在源码外执行观察代码并返回标准输出。"""
    # 1. 每次建立独立的安装目录和宿主工作目录。
    with tempfile.TemporaryDirectory(dir=chapter_dir) as directory:
        root = Path(directory)
        local_wheel = root / wheel_name
        local_wheel.write_bytes(wheel_bytes)
        target = root / "installed"
        outside = root / "outside"
        outside.mkdir()
        for name in ("plugin_host.py", "test_plugin_host.py"):
            shutil.copy2(chapter_dir / name, outside / name)

        # 2. 只安装本地 wheel，不写共享环境或安装字节码。
        subprocess.run(
            [sys.executable, "-B", "-m", "pip", "install", str(local_wheel),
             "--target", str(target), "--no-index", "--no-deps",
             "--no-compile", "--no-cache-dir", "--disable-pip-version-check"],
            cwd=outside, env=child_env, capture_output=True,
            encoding="utf-8", check=True, timeout=60,
        )
        installed_env = {**child_env, "PYTHONPATH": str(target)}
        prelude = (
            "import sys\nfrom pathlib import Path\n"
            "import importlib.metadata\nimport plugin_host\n"
            "target = Path(sys.argv[1])\n"
        )
        result = subprocess.run(
            [sys.executable, "-B", "-c", prelude + source, str(target)],
            cwd=outside, env=installed_env, capture_output=True,
            encoding="utf-8", check=True, timeout=60,
        )
    assert not root.exists()
    return result.stdout

# 后续单元调用此函数才真正安装；本单元只定义复用的实验步骤。

## 8 只发现安装目标中的元数据

importlib.metadata.distributions(path=[target]) 把默认文件系统元数据查找范围限制到安装目标；这里实际传入路径字符串。随后从每个 Distribution 的 entry\_points 中 select(group=...)，得到本组记录。

宿主 discover\_plugins 使用这条路径，要求提供本组入口的分发包具有非空 Name 和 Version，再交给 index\_entries 统一校验。它不调用 EntryPoint.load，因此尚未执行这些插件模块。全局 entry\_points 查询面向当前环境，本实验不使用它。

| API | 中文名称／含义 |
| --- | --- |
| distributions | 查找分发包元数据 |
| Distribution.entry\_points | 一份分发包登记的入口记录 |
| EntryPoints.select | 按组等属性筛选记录 |
| EntryPoint.load | 导入模块并解析入口对象 |

限定元数据路径不会改变 load 的导入搜索规则，因此仍需子进程 PYTHONPATH 与后面的实际导入位置检查。这是实验范围控制，不是安全隔离。

In [8]:
discovery_source = """
distributions = list(importlib.metadata.distributions(path=[str(target)]))
assert len(distributions) == 1
distribution = distributions[0]
assert distribution.metadata["Name"] == "study-plugin-demo"
assert distribution.version == "0.1.0"
entries = plugin_host.discover_plugins(target)
print(distribution.metadata["Name"], distribution.version)
print(sorted(entries))
assert set(entries) == {"upper", "broken", "reject", "dependency"}
assert "study_formatters" not in sys.modules
print("发现后尚未导入插件包")
"""
print(run_installed(discovery_source), end="")
# 只读取刚安装的分发元数据；四条记录均可发现，错误入口尚未被加载。

study-plugin-demo 0.1.0
['broken', 'dependency', 'reject', 'upper']
发现后尚未导入插件包


## 9 元数据规则和名称冲突

入口点规范由使用者决定跨分发包重名时的处理方式。本宿主采用立即失败策略：组内重名直接报错，绝不让发现顺序决定覆盖谁。先完成整张记录表的检查，之后才能加载选择的入口。

本宿主进一步约定：名称以小写英文字母开头，后续只含小写字母、数字或连字符；对象引用必须包含模块路径和属性路径，不接受 extras 或省略属性。这些是本例的接口规则，比通用入口点规范更窄。

下面构造两条相同名称的 EntryPoint，模拟来自不同分发包的冲突；它们不是本次真实安装的第二个分发包，也不会被加载。

In [9]:
collision_source = """
records = [
    importlib.metadata.EntryPoint(
        name="same", value=value, group=plugin_host.GROUP
    )
    for value in ("study_formatters.good:format_text", "math:sqrt")
]
for ordered in (records, list(reversed(records))):
    try:
        plugin_host.index_entries(ordered)
    except plugin_host.PluginError as exc:
        assert "重名" in str(exc)
        print(str(exc))
    else:
        raise AssertionError("应拒绝重名，不能静默覆盖")
assert "study_formatters" not in sys.modules
"""
print(run_installed(collision_source), end="")
# 两种顺序都应拒绝，元数据检查没有触发目标模块的导入。

发现：重名 'same'：'study_formatters.good:format_text' 与 'math:sqrt'
发现：重名 'same'：'math:sqrt' 与 'study_formatters.good:format_text'


## 10 加载后才得到入口对象

通过名称选中记录后，load\_plugin 调用 EntryPoint.load，并用 callable 检查对象。load 会按 value 导入模块、逐层读取属性；此时可以执行模块顶层代码，还没有把业务文本交给入口函数。

正常入口加载后检查模块的 \_\_file\_\_ 位于 target，确认使用的是安装产物。broken 虽然拥有合法的元数据引用，但其对象是整数，必须在加载边界拒绝。

In [10]:
load_source = """
entries = plugin_host.discover_plugins(target)
formatter = plugin_host.load_plugin(entries["upper"])
module = sys.modules["study_formatters.good"]
assert Path(module.__file__).resolve().is_relative_to(target.resolve())
print("正常入口可调用：", callable(formatter))
try:
    plugin_host.load_plugin(entries["broken"])
except plugin_host.PluginError as exc:
    assert "不可调用" in str(exc)
    print(str(exc))
else:
    raise AssertionError("整数不能成为文本处理入口")
"""
print(run_installed(load_source), end="")
# 正常对象来自临时安装目录；broken 不能因为 load 找到了对象就通过。

正常入口可调用： True
加载 'broken'：入口对象不可调用


## 11 区分缺少插件与插件内部导入失败

没有找到插件名称属于发现或选择阶段的问题；选中入口后出现 ModuleNotFoundError，则需要继续查看异常的 name。

如果缺失名称等于入口模块或是它的父包，宿主报告“目标导入路径缺失”；如果缺失名称是另一个模块，则报告“插件内部导入失败”。本例同时输出缺失名称，并用 raise ... from exc 保留原始原因，避免把依赖问题误报成“插件未安装”。这里 exc 表示捕获的原异常对象。

这个名称比较用于常规导入错误的诊断，不能证明插件代码一定如何运行；插件也可以自己抛出异常。完整原因链仍是排查依据。其他 ImportError 或 AttributeError 补充加载阶段信息，未约定的内部异常直接传播。

In [11]:
missing_source = """
entries = plugin_host.discover_plugins(target)
assert "absent" not in entries
print("未登记名称：absent")
# 这条记录是模拟配置，目标模块确实不存在。
missing_entry = importlib.metadata.EntryPoint(
    name="absent", value="study_formatters.absent:format_text",
    group=plugin_host.GROUP,
)
cases = [
    (missing_entry, "study_formatters.absent", "目标导入路径缺失"),
    (entries["dependency"],
     "study_plugin_demo_intentionally_missing_dependency", "插件内部导入失败"),
]
for entry, missing_name, message in cases:
    try:
        plugin_host.load_plugin(entry)
    except plugin_host.PluginError as exc:
        assert message in str(exc)
        assert isinstance(exc.__cause__, ModuleNotFoundError)
        assert exc.__cause__.name == missing_name
        print(entry.name, message, exc.__cause__.name)
    else:
        raise AssertionError("本例应在加载阶段失败")
"""
print(run_installed(missing_source), end="")
# dependency 来自真实安装记录；没有为故意缺失的依赖执行 pip 安装。

未登记名称：absent
absent 目标导入路径缺失 study_formatters.absent
dependency 插件内部导入失败 study_plugin_demo_intentionally_missing_dependency


## 12 调用、结果检查与失败策略

call\_plugin 把一个字符串位置参数交给插件，检查返回值也是字符串。本例只把 TypeError 和 ValueError 转换为带插件名的 PluginError，并保留原因链；这两类异常也可能来自插件内部，不能仅凭类型断定是调用者参数有错。

发现冲突或选中的插件失败时立即中止当前流程，不返回空字符串冒充成功。下面的 try/except 仅用于观察故意设计的反例，并不意味着宿主会自动跳过故障插件。

In [12]:
call_source = """
entries = plugin_host.discover_plugins(target)
formatter = plugin_host.load_plugin(entries["upper"])
for text, expected in [("Py插件", "PY插件"), ("", "")]:
    result = plugin_host.call_plugin("upper", formatter, text)
    assert result == expected
    print(repr(result))
reject = plugin_host.load_plugin(entries["reject"])
try:
    plugin_host.call_plugin("reject", reject, "hello")
except plugin_host.PluginError as exc:
    assert isinstance(exc.__cause__, ValueError)
    print(str(exc), type(exc.__cause__).__name__)
else:
    raise AssertionError("受控拒绝不能变成成功输出")
"""
print(run_installed(call_source), end="")
# 正常文本和空文本均成功；reject 在调用阶段失败并保留 ValueError。

'PY插件'
''
调用 'reject'：演示插件拒绝文本：'hello' ValueError


### 12.1 可调用性不等于完整接口验证

需要两个参数的函数和返回整数的函数都可以通过 callable。宿主仍要处理调用失败并校验结果；参数与返回类型标注不能取代这些边界检查。即使使用可在运行时检查的 Protocol，也只检查所需属性是否存在，不检查完整类型签名。

In [13]:
contract_source = """
def needs_two(text: str, suffix: str) -> str:
    return text + suffix

for name, candidate in [("two", needs_two), ("length", len)]:
    assert callable(candidate)
    try:
        plugin_host.call_plugin(name, candidate, "hello")
    except plugin_host.PluginError as exc:
        if name == "two":
            assert isinstance(exc.__cause__, TypeError)
        else:
            assert "返回值" in str(exc)
        print(name, str(exc))
    else:
        raise AssertionError("不符合接口的对象不能通过调用边界")
"""
print(run_installed(contract_source), end="")
# 两者都可调用，但参数数量和返回类型分别违背宿主约定。

two 调用 'two'：needs_two() missing 1 required positional argument: 'suffix'
length 调用 'length'：返回值必须是字符串


## 13 可信插件与行为检查

插件在导入和调用时拥有所在进程可用的代码执行能力；名称、类型和元数据校验都不会把它变成沙箱。子进程可以让本实验的模块缓存随进程结束而消失，但它仍使用当前用户权限。本章只加载自己编写并检查过的插件。

配套测试用明确输入检查非法名称、引用规则、双向重名顺序、其他接口组、属性缺失、错误签名、错误返回值与未知异常传播。EntryPoint 模拟记录只用于这些算法边界；安装与真实入口加载已由前面的实验单独检查。

In [14]:
test_source = """
import pytest

status = pytest.main([
    "-q", "-p", "no:cacheprovider", "--basetemp", "test-temp",
    "test_plugin_host.py",
])
assert status == 0
"""
print(run_installed(test_source), end="")
assert "study_formatters" not in sys.modules
# 17 个宿主测试应通过；测试工作目录及临时数据随后一起清理。
# Notebook 没有导入插件包，也没有向共享环境安装这个分发包。

.................                                                        [100%]
17 passed in 0.05s


## 本章小结

（1）动态导入沿用模块搜索与缓存；查找缓存失效不等于重新执行已有模块。

（2）入口点把接口组、插件名和对象引用写入分发元数据。先发现并校验，再选择、加载和调用。

（3）接口约定、名称冲突策略和错误边界由宿主决定。可调用检查不足以证明签名或行为正确，异常转换要保留原因链。

自查：如果插件已经被发现，却在 load 时缺少另一个模块，为什么不能直接提示“没有安装插件”？

## 练习

（1）先预测下面三个布尔结果，再运行核对。分别解释正常入口、不可调用入口和分发包名称的含义。核对标准是预测与实际值一致，并能指出入口对象是否可调用只能在什么阶段检查。

In [15]:
print(entry_mapping["upper"] == "study_formatters.good:format_text")
print(entry_mapping["broken"] == "study_formatters.bad:format_text")
print(project_config["project"]["name"] == "study_formatters")
# 先写预测，再核对配置；不要用修改配置的方式匹配预测。

True
False
False


（2）只在临时源码副本中添加 lower 入口，把它指向新增函数，返回文本的小写形式；重新构建并安装自己的 wheel。成功标准：通过指定 target 的元数据发现 lower，在源码目录外加载；输入 Py插件 得到 py插件，空字符串仍得到空字符串，导入位置在 target 内。

继续确认原配套配置仍只有原来的四条记录，并确认临时源码、wheel 与安装目录已删除。

In [16]:
exercise_inputs = ["Py插件", ""]
# 复制配置与 src 后添加函数及入口；沿用临时构建、安装与子进程检查。
# 本题需重新构建，不可只改源码后继续测试旧 wheel。

（3）调用 run\_installed，构造两个插件函数：一个抛出 ValueError，另一个抛出 LookupError。前者应成为 PluginError，且 \_\_cause\_\_ 是原 ValueError 对象；后者应按原 LookupError 对象传播。每个反例都用具体 except 和 else 中的失败断言检查，不返回默认值。

再构造两个相同名称、不同引用的 EntryPoint，检查正序和倒序都拒绝；明确说明这是模拟跨分发包冲突，不能把它表述为安装了两个分发包。

In [17]:
exercise_failure_types = (ValueError, LookupError)
# 在子进程中保存将要抛出的异常对象，再用 is 检查传播或原因链身份。
# 重名记录交给 index_entries；校验结束前不要调用 load。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（3.12） | [动态导入](https://docs.python.org/3.12/library/importlib.html#importlib.import_module)、[查找缓存失效](https://docs.python.org/3.12/library/importlib.html#importlib.invalidate_caches)、[重新加载及外部引用限制](https://docs.python.org/3.12/library/importlib.html#importlib.reload)；[模块缓存](https://docs.python.org/3.12/reference/import.html#the-module-cache)、[模块加载与执行](https://docs.python.org/3.12/reference/import.html#loading)、[命名空间包](https://docs.python.org/3.12/reference/import.html#namespace-packages)；[限定路径枚举模块](https://docs.python.org/3.12/library/pkgutil.html#pkgutil.iter_modules)；[入口查询与加载](https://docs.python.org/3.12/library/importlib.metadata.html#entry-points)、[分发包](https://docs.python.org/3.12/library/importlib.metadata.html#distributions)、[元数据路径上下文](https://docs.python.org/3.12/library/importlib.metadata.html#extending-the-search-algorithm)；[Callable 参数与返回值](https://docs.python.org/3.12/library/typing.html#annotating-callable-objects)、[Protocol](https://docs.python.org/3.12/library/typing.html#typing.Protocol)、[运行时协议检查限制](https://docs.python.org/3.12/library/typing.html#typing.runtime_checkable)、[callable 的限制](https://docs.python.org/3.12/library/functions.html#callable)；[ImportError 的 name](https://docs.python.org/3.12/library/exceptions.html#ImportError)、[ModuleNotFoundError](https://docs.python.org/3.12/library/exceptions.html#ModuleNotFoundError)、[异常原因链](https://docs.python.org/3.12/reference/simple_stmts.html#the-raise-statement)；[子进程](https://docs.python.org/3.12/library/subprocess.html#subprocess.run)、[PYTHONPATH](https://docs.python.org/3.12/using/cmdline.html#envvar-PYTHONPATH)、[禁止字节码写入](https://docs.python.org/3.12/using/cmdline.html#envvar-PYTHONDONTWRITEBYTECODE)、[临时目录](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryDirectory)、[复制目录](https://docs.python.org/3.12/library/shutil.html#shutil.copytree)、[TOML 读取](https://docs.python.org/3.12/library/tomllib.html#tomllib.load)；[正则完整匹配](https://docs.python.org/3.12/library/re.html#re.fullmatch)、[标识符检查](https://docs.python.org/3.12/library/stdtypes.html#str.isidentifier)、[字符串大写](https://docs.python.org/3.12/library/stdtypes.html#str.upper)、[字符串小写](https://docs.python.org/3.12/library/stdtypes.html#str.lower)、[平方根](https://docs.python.org/3.12/library/math.html#math.sqrt)。 |
| PyPA 打包指南与规范 | [三种插件发现方式](https://packaging.python.org/en/latest/guides/creating-and-discovering-plugins/#using-naming-convention)、[命名空间发现](https://packaging.python.org/en/latest/guides/creating-and-discovering-plugins/#using-namespace-packages)、[元数据发现](https://packaging.python.org/en/latest/guides/creating-and-discovering-plugins/#using-package-metadata)；[入口点数据模型、组命名和冲突策略](https://packaging.python.org/en/latest/specifications/entry-points/#data-model)、[入口元数据文件](https://packaging.python.org/en/latest/specifications/entry-points/#file-format)、[命令入口的特殊用途](https://packaging.python.org/en/latest/specifications/entry-points/#use-for-scripts)；[构建配置](https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#declaring-the-build-backend)、[Python 版本声明](https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#requires-python)。 |
| GitHub 上的官方与维护者源码 | CPython v3.12.14 的 [importlib.metadata 实现](https://github.com/python/cpython/blob/v3.12.14/Lib/importlib/metadata/__init__.py)，定位 distributions、Distribution.discover、DistributionFinder.Context.path、EntryPoint.load，核对 path 参数如何传入查找及加载如何导入对象；setuptools v83.0.0 的 [pyproject 配置](https://github.com/pypa/setuptools/blob/v83.0.0/docs/userguide/pyproject_config.rst)，定位 example-pyproject-config 与 packages.find；[入口点文档](https://github.com/pypa/setuptools/blob/v83.0.0/docs/userguide/entry_point.rst)，定位 Entry Points for Plugins，核对自定义组及同一分发包的多个入口。 |
| build 官方文档（1.6.1） | [构建 wheel、关闭隔离和构建依赖检查](https://build.pypa.io/en/stable/reference/cli.html#python--m-build)、[关闭隔离时不安装构建依赖](https://build.pypa.io/en/stable/reference/cli.html#dependency-check)。 |
| pip 官方文档 | [目标目录安装](https://pip.pypa.io/en/stable/cli/pip_install/#install-target)、[不查询索引](https://pip.pypa.io/en/stable/cli/pip_install/#install-no-index)、[不安装依赖](https://pip.pypa.io/en/stable/cli/pip_install/#install-no-deps)、[不编译字节码](https://pip.pypa.io/en/stable/cli/pip_install/#install-no-compile)。 |
| pytest 官方文档 | 本章运行 pytest 9.1.1；[参数化](https://docs.pytest.org/en/stable/how-to/parametrize.html#pytest-mark-parametrize-parametrizing-test-functions)、[预期异常](https://docs.pytest.org/en/stable/how-to/assert.html#assertions-about-expected-exceptions)、[关闭插件自动加载](https://docs.pytest.org/en/stable/reference/reference.html#envvar-PYTEST_DISABLE_PLUGIN_AUTOLOAD)。 |
| Python PEP 文档 | [PEP 578：Why Not A Sandbox](https://peps.python.org/pep-0578/#why-not-a-sandbox)，辅助说明 Python 内部检查机制与安全沙箱不是一回事。 |